# 🗄️ 4. Text2SQL: 시멘틱 레이어, 스키마 링킹, 값 연결

앞선 `3_Vector_and_Graph_RAG.ipynb`에서는 사내 규정이나 아키텍처 백서 같은 **비정형 문서**를 마크다운 청크와 지식 그래프로 인덱싱하는 방법을 살펴보았습니다.  
반면 기업의 실시간 예산 집행액, 결재 대기 전표, 임직원 인사 정보, GPU 클러스터 가동률 같은 **정형 수치 데이터**는 관계형 데이터베이스(RDB / ERP)에 저장되어 있습니다.

표 데이터를 텍스트 청크로 잘라 벡터 DB에 넣으면 상위 일부(`top-k`) 조각만 검색되므로 **전체 합계(`SUM`)나 비율 계산이 틀리고**, 전표 결재 상태가 바뀔 때마다 다시 인덱싱해야 합니다. 따라서 정형 데이터는 **자연어 질문을 SQL 쿼리로 변환해 DB 엔진이 직접 집계하도록 맡기는 Text2SQL** 방식이 적합합니다.

이번 노트북에서는 먼저 사내 ERP 데이터베이스의 구조를 살펴본 뒤, 물리적 `CREATE TABLE` 스키마만 LLM에 주었을 때 왜 오답이 발생하는지 실험합니다. 이어서 이를 해결하는 세 가지 핵심 기법인 **시멘틱 레이어(Semantic Layer)**, **스키마 링킹(Schema Linking)**, **값 연결(Value Linking)** 과 **자가 수정(Self-Healing) 실행 워크플로우(`rag/sql/`)** 를 차례대로 구현하고 검증합니다.

---

### 🎯 학습 목표
1. 원시 DDL(`CREATE TABLE`)만 제공할 때 발생하는 값 불일치(`0행` 반환)와 업무 기준 모호성 문제를 재현하고 원인을 파악합니다.
2. DB에서 자동 추출한 물리 구조 위에 용어 정의·집계 규칙·지표 공식을 결합하는 **시멘틱 레이어(`DatabaseProfile`)** 의 역할을 이해합니다.
3. 필요한 테이블과 상위 외래키(FK) 조인 경로만 추려내는 **스키마 링킹(`SchemaRetriever`)** 과 일상 표현을 실제 셀 값에 매칭하는 **값 연결(`ValueLinker`)** 의 동작 원리를 익힙니다.
4. 읽기 전용 안전장치(`executor.py`)와 오류·`0행` 자가 수정 및 되묻기 워크플로우(`Text2SQL`)를 비정형 규정 검색 도구와 결합해 복합 감사를 수행합니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 모듈 / 기법 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 프로젝트 루트 탐색, `.env`, LLM 및 임베딩 모델 초기화 |
| **Part 1** | 실습 DB 구조 이해와 원시 DDL 방식의 한계 | `enterprise_erp.db` 7개 테이블 구조, 실패 실험 2종, 3대 해결 기법 개요 |
| **Part 2** | 시멘틱 레이어 (Semantic Layer) | `DatabaseProfile` · 물리 스키마(`PRAGMA`) + 업무 정의(`semantic_layer.yaml`) 결합 |
| **Part 3** | 스키마 링킹 (Schema Linking) | `SchemaRetriever` · 테이블/지표 카드 검색 및 상위(부모) 방향 FK 확장 |
| **Part 4** | 값 연결 (Value Linking) | `ValueLinker` · 선택적 값 색인, 최장 일치 매칭(`'정수석'` vs `'수석'`), 누락 테이블 역보강 |
| **Part 5** | 읽기 전용 가드레일과 자가 수정 워크플로우 | `validate_sql` · `run_readonly_query` · `Text2SQL` LangGraph 재검증 및 되묻기 분기 |
| **Part 6** | 비정형 규정 × 정형 ERP 에이전트 도구 결합 | `search_company_policy` + `search_enterprise_erp(context=...)` 여비 규정 위반 전표 감사 |
| **Part 7** | 정리 및 다음 단계 | Text2SQL 설계 원칙 요약 및 `5_MCP_and_Skills.ipynb` 연결 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`), LLM(`gemini-3.8-flash`), 임베딩 모델(`text-embedding-3-large`)을 초기화합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)
print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. LLM 및 임베딩 모델 초기화
from langchain_openai import OpenAIEmbeddings
from app.utils import init_chat_model, normalize_content

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print(f"✅ LLM 초기화 완료: {llm.model}")

---
## 💥 Part 1. 원시 DDL만 주면 왜 실패할까?

### 1.1 실습 대상 데이터베이스 미리보기: (주)넥스트AI 사내 ERP (`enterprise_erp.db`)

본격적인 실험에 앞서, 우리가 조회할 `app/database/enterprise_erp.db` 안에 어떤 데이터가 어떻게 나뉘어 저장되어 있는지 먼저 살펴보겠습니다. 이 데이터베이스는 현업 ERP와 유사하게 **7개의 정규화된 테이블**로 이루어져 있습니다.

| 구분 | 테이블명 | 행 수 | 담고 있는 내용 및 주요 컬럼 |
|:---|:---|---:|:---|
| **조직·인사** | `departments` | 7행 | 부서 마스터 — 부서코드(`dept_code`), 정식 부서명(`dept_name`: `'클라우드운영팀'` 등), 약칭(`dept_alias`: `'클운팀'` 등), 연간 총예산(`annual_budget_krw`) |
| **조직·인사** | `employees` | 40행 | 임직원 마스터 — 사번(`emp_id`), 성명(`name`: `'김철수'`, `'정수석'` 등), 직급(`rank`: `'전무'`, `'수석'` 등), 소속 부서코드, 직속 상관 사번(`manager_emp_id`) |
| **부서 예산** | `department_budgets` | 20행 | **부서별 연간 운영 예산** — 비목(`category`)별 **편성액(`allocated_krw`)과 누적 집행액(`spent_krw`)이 한 테이블에 함께 저장됨** |
| **전사 과제** | `projects` | 6행 | **전사 전략 프로젝트(`P-01`~`P-06`)** — 과제명, 주관부서, PM 사번, **승인 예산(`approved_budget_krw`)만 있고 집행액 컬럼은 없음** |
| **지출 전표** | `expense_transactions` | 135행 | **건별 지출 품의/전표** — 프로젝트(`project_id`), 기안자 사번, 금액(`amount_krw`), 결재 상태(`approval_status`: **`'승인완료'` · `'결재대기'` · `'반려'`**) 기록 |
| **투입·인프라** | `project_members` | 52행 | 프로젝트별 투입 인력 사번(`emp_id`), 담당 역할, 투입률(`allocation_pct`) |
| **투입·인프라** | `gpu_cluster_allocations` | 10행 | 프로젝트별 할당 GPU 기종, 평균 사용률(`avg_utilization_pct`), 월 임차 비용 |

테이블 간의 외래키(FK) 연결 구조는 다음과 같습니다.

```mermaid
erDiagram
    departments ||--o{ department_budgets : "부서예산"
    departments ||--o{ employees : "소속부서"
    employees ||--o{ employees : "직속상관"
    employees ||--o{ project_members : "투입인력"
    projects ||--o{ project_members : "투입과제"
    employees ||--o{ expense_transactions : "기안·결재"
    projects ||--o{ expense_transactions : "지출집행"
    departments ||--o{ projects : "주관부서"
    employees ||--o{ projects : "PM·승인"
    projects ||--o{ gpu_cluster_allocations : "할당과제"
    departments ||--o{ gpu_cluster_allocations : "관리부서"
```

> 📌 **실험 전 눈여겨볼 두 가지 데이터 구조 특징**
> 1. **부서명 저장 방식**: `departments` 테이블에는 정식 명칭(`'클라우드운영팀'`)과 사내 약칭(`'클운팀'`)이 저장되어 있으며, 흔히 줄여 부르는 `'클라우드팀'`이라는 문자열은 들어 있지 않습니다.
> 2. **두 갈래로 나뉜 예산 구조**: 이 회사에는 성격이 다른 두 가지 예산이 공존합니다.
>    - **부서 운영 예산(`department_budgets`)**: 한 테이블 안에 편성액(`allocated_krw`)과 집행액(`spent_krw`) 컬럼이 나란히 들어 있어 바로 비율을 계산할 수 있습니다.
>    - **전략 프로젝트 예산(`projects` + `expense_transactions`)**: `projects` 테이블에는 승인 예산(`approved_budget_krw`)만 적혀 있고, 실제로 쓴 돈은 지출 전표 테이블(`expense_transactions`)에서 결재가 완료(`'승인완료'`)된 건들을 `JOIN`해 합산해야만 알 수 있습니다.

아래 코드 셀을 실행해 실제 DB에 저장된 부서 명칭과 두 예산 테이블(`department_budgets` vs `projects`)의 원시 `CREATE TABLE` 문장(DDL)을 직접 확인해 보겠습니다.

In [ ]:
from rag.sql.executor import connect_readonly

ERP_DB_PATH = "app/database/enterprise_erp.db"
SEMANTIC_LAYER_PATH = "data/enterprise_erp_sql/semantic_layer.yaml"

conn = connect_readonly(ERP_DB_PATH)
ddl_rows = conn.execute(
    "SELECT name, sql FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY rowid"
).fetchall()

print("📊 [1. 사내 ERP 7개 테이블 행 수]")
for table_name, _ in ddl_rows:
    cnt = conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    print(f"  - {table_name:<26} : {cnt:>4}행")

print("\n🏢 [2. departments 테이블에 실제 저장된 부서명·약칭 확인]")
for code, name, alias, budget in conn.execute(
    "SELECT dept_code, dept_name, dept_alias, annual_budget_krw FROM departments"
).fetchall():
    print(f"  - {code} | 정식명: {name:<10} | 약칭: {alias:<5} | 연간예산: {budget:>13,}원")

ddl_map = dict(ddl_rows)
conn.close()

print("\n" + "=" * 75)
print("📄 [3. DB가 기본 제공하는 원시 DDL 비교: department_budgets vs projects]")
print("   👉 department_budgets에는 spent_krw(집행액) 컬럼이 있지만, projects에는 없습니다.\n")
print(ddl_map["department_budgets"])
print("-" * 75)
print(ddl_map["projects"])

### 1.2 실험: 원시 DDL(`CREATE TABLE`)만 프롬프트에 넣었을 때의 동작 확인

가장 단순한 Text2SQL 구현 방식은 위에서 본 `CREATE TABLE` 문장 7개를 그대로 프롬프트에 넣고 LLM에게 SQL 작성을 요청하는 것입니다.  
이 방식에 현업에서 흔히 묻는 두 가지 질문을 던지면 어떤 SQL과 결과가 나오는지 직접 확인해 보겠습니다.

1. **질문 1 (일상 표현과 DB 실제 셀 값의 차이)**: `"클라우드팀의 연간 운영 예산과 비목별 편성액을 알려줘."`
   - 실제 DB에는 `'클라우드운영팀'`(약칭 `'클운팀'`)으로 연간 8억 5,000만 원과 4개 비목 예산이 등록되어 있지만, `CREATE TABLE` 문장 안에는 실제 부서명 값들이 적혀 있지 않습니다.
2. **질문 2 (두 종류의 예산 원장과 모호한 기준)**: `"예산 집행률이 80% 이상인 항목을 모두 알려줘."`
   - 실제 DB에는 **부서 운영 예산(`department_budgets`)** 에도 집행률 80% 이상 비목(4건)이 있고, **전략 프로젝트 예산(`projects` + `expense_transactions`)** 에도 집행률 80% 이상 과제(`P-02` 86.0%, `P-05` 85.0%)가 있습니다. 원시 DDL만 받은 모델이 어떻게 판단하는지 살펴봅니다.

In [ ]:
from pydantic import BaseModel, Field
from rag.sql import run_readonly_query

class RawSQLOutput(BaseModel):
    sql: str = Field(description="실행할 단일 SQLite SELECT 문")

naive_sql_llm = llm.with_structured_output(RawSQLOutput)
all_raw_ddls = "\n\n".join(sql for _, sql in ddl_rows)

def run_naive_text2sql(question: str) -> None:
    prompt = (
        "다음 SQLite CREATE TABLE 스키마만 보고 질문에 답하는 단일 SELECT SQL을 작성하세요.\n\n"
        f"{all_raw_ddls}\n\n질문: {question}"
    )
    draft = naive_sql_llm.invoke(prompt)
    res = run_readonly_query(ERP_DB_PATH, draft.sql)
    print(f"❓ 질문: {question}")
    print(f"📝 생성된 SQL:\n  {draft.sql}")
    print(f"📊 실행 결과 ({res.summary()}):\n{res.to_markdown()}\n" + "-" * 75)

run_naive_text2sql("클라우드팀의 연간 운영 예산과 비목별 편성액을 알려줘.")
run_naive_text2sql("예산 집행률이 80% 이상인 항목을 모두 알려줘.")

### 1.3 실험 결과 진단: 왜 문법 오류 없이 틀린 답이 나올까?

위 두 실험 모두 **SQL 문법 오류는 발생하지 않았습니다.** 하지만 실제 DB에 들어 있는 정답과 비교해 보면 두 결과 모두 사용자의 의도를 벗어난 **'조용한 오답(Silent Wrong Answer)'** 입니다. 에러 메시지 대신 그럴듯한 표나 `0행`이 반환되기 때문에, 결과를 받는 사람은 오류 여부를 알아채기 어렵습니다.

| 실험 질문 | 실제 DB 기준 원래 나왔어야 할 정상 결과 | 원시 DDL만 본 모델의 실제 오판과 원인 |
|:---|:---|:---|
| **질문 1**<br/>`"클라우드팀의 연간 운영 예산과 비목별 편성액을 알려줘"` | `'클라우드팀'`을 정식 명칭 **`'클라우드운영팀'`** 으로 연결하여 총예산 8.5억 원 및 4개 비목(`GPU클러스터임차` 4억, `외주용역` 2.5억, `국내출장` 1.2억, `해외출장` 0.8억)을 출력해야 함 | 셀에 저장된 실제 문자열을 모르기 때문에 질문 표현 그대로 `WHERE dept_name = '클라우드팀'`을 실행하여 **`0행`을 반환**함 ➔ 사용자가 *"클라우드팀은 편성 예산이 없다"* 고 오해하게 됨 |
| **질문 2**<br/>`"예산 집행률이 80% 이상인 항목을 모두 알려줘"` | DB에 두 종류의 예산 원장이 공존하므로, **사용자에게 "부서 운영 예산과 전략 프로젝트 예산 중 어느 쪽을 조회할까요?"라고 되묻거나** 양쪽 기준을 구분해 안내해야 함 | `department_budgets`에는 `allocated_krw`와 `spent_krw`가 한눈에 보이지만 `projects`에는 집행액 컬럼이 없자, 확인 절차 없이 **계산하기 쉬운 부서 운영 예산 한쪽만 임의로 조회**함 ➔ 집행률 경보 대상인 전략 프로젝트 **`P-02`(86.0%), `P-05`(85.0%)가 통째로 누락**됨 |

이처럼 실무 Text2SQL의 핵심 과제는 SQL 문법 생성 자체보다 **"사용자의 일상적인 업무 표현을 DB의 올바른 테이블 · 계산 기준 · 실제 셀 값에 정확히 연결하는 것"** 에 있습니다. 이를 해결하는 세 가지 핵심 기법은 다음과 같이 역할을 나누어 동작합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    Q["사용자 자연어 질문"]
    SL["1. 시멘틱 레이어 (Semantic Layer)<br/>물리 스키마 + 용어 뜻 · 집계 규칙 · 공식<br/>('DB용 업무 인수인계서')"]

    subgraph LINK["질문과 DB 요소 연결 (사전 색인 활용)"]
        direction TB
        S["2. 스키마 링킹 (Schema Linking)<br/>질문에 필요한 테이블 · 지표 카드 선별<br/>+ 참조 부모 테이블(FK) 자동 추가"]
        V["3. 값 연결 (Value Linking)<br/>질문 속 약어 · 고유명사를<br/>DB 실제 저장 문자열에 매칭"]
    end

    G["SQL 생성 및 안전 실행<br/>(되묻기 분기 + 자가 수정 루프)"]

    Q --> LINK
    SL -.->|"카드 · 허용값 색인 제공"| LINK
    S ==>|"FROM · JOIN · SELECT 재료"| G
    V ==>|"WHERE 조건 문자열 재료"| G

    style SL fill:#fff8e1,stroke:#f57f17,stroke-width:1.5px
    style S fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style V fill:#fff3e0,stroke:#fb8c00,stroke-width:1.5px
    style G fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

| 핵심 기법 | 해결하려는 문제 | 직관적 비유 | SQL 작성에서 맡는 역할 | 구현 모듈 (`rag/sql/`) |
|:---|:---|:---|:---|:---|
| **1. 시멘틱 레이어**<br/>*(Semantic Layer)* | 컬럼 이름과 타입만으로는 회사의 용어 정의나 집계 공식(예: 실집행액 산정 기준)을 알 수 없는 문제 | **신입 분석가용 업무 인수인계서**<br/>*(숫자 장부에 뜻과 공식을 적어 줌)* | **`SELECT` 계산식 및 해석 기준 제공**<br/>(승인완료 건만 합산, 2종 예산 구분 등) | `DatabaseProfile`<br/>(`semantic_layer.py`) |
| **2. 스키마 링킹**<br/>*(Schema Linking)* | 수십~수백 개 테이블을 다 넣으면 혼선이 생기고, 적게 넣으면 `JOIN`할 마스터 테이블이 빠지는 문제 | **필요한 장부와 코드표만 세트로 꺼내기**<br/>*(연결된 상위 기준 장부까지 챙김)* | **`FROM` · `JOIN` 대상 테이블 선별**<br/>(카드 하이브리드 검색 + 부모 FK 확장) | `SchemaRetriever`<br/>(`schema_retriever.py`) |
| **3. 값 연결**<br/>*(Value Linking)* | 약어나 일상 표현을 그대로 `WHERE` 절에 쓰면 글자 불일치로 `0행`이 나오는 문제 | **사내 약어 → 실제 저장 코드 번역표**<br/>*(실제 셀에 적힌 글자 그대로 매칭)* | **`WHERE` 조건의 정확한 문자열 제공**<br/>(`'클운팀'` → `dept_name = '클라우드운영팀'`) | `ValueLinker`<br/>(`value_linker.py`) |

이어지는 Part 2부터 Part 4까지 세 기법이 각각 어떤 문제를 어떻게 해결하는지 실제 코드와 데이터로 확인해 보겠습니다.

---
## 📖 Part 2. 시멘틱 레이어 (Semantic Layer): 물리 스키마에 업무 정의와 공식 입히기

### 1) 어떤 문제를 해결하기 위한 것인가?

DB의 원시 스키마(`CREATE TABLE`)에는 컬럼의 영문 이름과 자료형(`INTEGER`, `TEXT`)만 적혀 있을 뿐, **현업에서 쓰는 용어가 어떤 조건과 계산식을 뜻하는지**는 적혀 있지 않습니다.

실제 우리 DB에 저장된 **`P-02` 프로젝트(엔터프라이즈 Agentic RAG 플랫폼 구축, 승인 예산 4억 8,000만 원)** 의 지출 전표(`expense_transactions`) 데이터를 예로 들어 보겠습니다. `expense_transactions` 테이블에는 결재 진행 단계(`approval_status`)에 따라 세 종류의 전표가 한데 섞여 저장되어 있습니다.

| 결재 상태 (`approval_status`) | 의미 | `P-02` 전표 합계 금액 | 회사 회계상 '실집행액' 포함 여부 |
|:---:|:---|---:|:---:|
| **`'승인완료'`** | 결재가 최종 완료되어 실제 집행이 확정된 금액 | **4억 1,280만 원** | **✅ 포함** (집행률 **86.0%**) |
| **`'결재대기'`** | 실무자가 기안만 올리고 아직 결재 전인 금액 | 6,540만 원 | ❌ 제외 (미확정) |
| **`'반려'`** | 결재 과정에서 거절되어 지출되지 않은 금액 | 520만 원 | ❌ 제외 (집행 안 됨) |

재무 담당자가 *"P-02 과제의 실집행액과 예산 집행률을 알려줘"* 라고 물으면, 회사 기준상 확정된 **`'승인완료'` 전표만 더한 4억 1,280만 원(86.0%)** 이 나와야 합니다.  
하지만 이 업무 규칙을 알려주지 않으면 LLM은 `expense_transactions`의 금액 컬럼(`amount_krw`)을 조건 없이 모두 더하는 `SUM(amount_krw)` 쿼리를 작성하기 쉽습니다. 그러면 대기·반려 건까지 합산되어 **4억 8,340만 원(집행률 100.7%, 예산 초과)** 이라는 잘못된 수치가 보고됩니다.

### 2) 어떻게 작동하는가? (비유와 4계층 결합 원리)

갓 입사한 데이터 분석가에게 숫자만 적힌 엑셀 파일만 건네주면 집계 기준을 몰라 실수하지만, **'업무 인수인계서(용어 정의 · 포함/제외 조건 · 자주 쓰는 지표 공식)'** 를 함께 주면 정확한 수치를 산출할 수 있습니다.  
**시멘틱 레이어(Semantic Layer)** 는 BI 도구(dbt Semantic Layer, LookML 등)와 Text2SQL 시스템에서 공통으로 사용하는 계층으로, **DB의 물리적 테이블·컬럼 위에 현업의 용어 뜻·해석 규칙·계산 공식을 매핑해 둔 'DB용 업무 사전'** 입니다.

관리 부담을 줄이기 위해 `rag/sql/semantic_layer.py`의 `DatabaseProfile`은 DB 자체가 이미 알고 있는 물리 정보는 자동으로 읽어 오고, **사람만 아는 업무 규칙만 설정 파일(`semantic_layer.yaml`)에서 읽어 결합**합니다.

| 구성 계층 | 담는 내용 | 작성 주체 | 실제 적용 예시 (`enterprise_erp`) | 없을 때 발생하는 문제 |
|:---|:---|:---:|:---|:---|
| **① 물리 스키마** | 컬럼명, 자료형, PK, 외래키(FK), `CHECK` 허용값, 샘플 2행 | **DB 자동 추출**<br/>(`PRAGMA`) | `approval_status` 허용값 = `'승인완료'`, `'결재대기'`, `'반려'` | 존재하지 않는 컬럼을 쓰거나 조인 키를 틀림 |
| **② 데이터 사전** | 테이블·컬럼의 한글 의미와 현업 동의어 | 데이터 담당자<br/>(`YAML`) | `expense_transactions` 동의어 = `품의, 전표, 지출결의` | `"품의 내역"`이라고 물으면 어느 테이블인지 못 찾음 |
| **③ 업무 규칙** | 모호한 용어의 기본 해석 기준 (`global_rules`, `rules`) | 업무 담당자<br/>(`YAML`) | *"실집행액은 `approval_status = '승인완료'` 건만 합산"*, *"예산은 프로젝트/부서 2종 구분"* | 결재대기·반려 금액까지 합산하거나 한쪽 예산만 임의 조회 |
| **④ 지표 공식** | 물리 컬럼으로 없지만 자주 묻는 계산식·단위·필요 테이블 | 업무 담당자<br/>(`YAML`) | `budget_burn_rate` = `승인완료 합계 * 100.0 / approved_budget_krw` | 질문할 때마다 집행률 수식이나 소수점 처리가 달라짐 |

> 💡 **설계 기준 — 사내 규정 문서의 기준값은 시멘틱 레이어에 중복 기재하지 않습니다.**  
> 직급별 출장 숙박비 상한액(여비규정 제14조)이나 위임전결 금액 기준처럼 **비정형 사내 규정에 정의된 수치**는 규정 개정 주기에 따라 바뀝니다. 이런 기준값까지 DB YAML에 적어 두면 규정이 바뀔 때마다 양쪽을 고쳐야 하므로, 런타임에 에이전트가 규정 검색 도구로 조항을 먼저 읽은 뒤 `context`(참고 기준값) 인자로 넘기도록 역할을 분리합니다(Part 6에서 실습).

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    DB["enterprise_erp.db<br/>① 물리 스키마 자동 추출<br/>(컬럼 · PK/FK · 허용값 · 샘플 2행)"] --> PROF["DatabaseProfile.load()<br/>물리 구조 + 업무 사전 통합"]
    YAML["semantic_layer.yaml<br/>② 테이블·컬럼 설명 및 동의어<br/>③ 업무 규칙 · ④ 지표 계산 공식"] --> PROF

    PROF --> OV["profile.overview()<br/>7개 테이블 한 줄 요약 + 전사 공통 규칙<br/>(프롬프트 상단에 항상 포함)"]
    PROF --> TC["profile.table_card(name)<br/>테이블 상세 카드 7장<br/>(질문에 필요한 카드만 검색해 주입)"]
    PROF --> MC["profile.metric_card(key)<br/>지표 공식 카드 6장<br/>(질문에 필요한 공식만 검색해 주입)"]

    style DB fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style YAML fill:#fff8e1,stroke:#f57f17,stroke-width:1.5px
    style PROF fill:#ede7f6,stroke:#5e35b1,stroke-width:1.5px
    style OV fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

### 2.1 `DatabaseProfile` 로드 및 전사 요약(`overview`) 확인

`DatabaseProfile.load()`를 실행해 물리 DB와 `semantic_layer.yaml`이 결합된 결과를 확인해 보겠습니다. `overview()`는 전체 테이블의 한 줄 요약과 전사 공통 규칙(`global_rules`)을 담고 있어, 모든 SQL 생성 프롬프트 상단에 기본 컨텍스트로 제공됩니다.

In [ ]:
from rag.sql import DatabaseProfile

profile = DatabaseProfile.load(SEMANTIC_LAYER_PATH, db_path=ERP_DB_PATH)

print(f"✅ 로드 완료: {profile.title} ({profile.dialect})")
print(f"  - 테이블 수 : {len(profile.tables)}개 -> {list(profile.tables.keys())}")
print(f"  - 지표 수   : {len(profile.metrics)}개 -> {list(profile.metrics.keys())}")
print("\n" + "=" * 75)
print(profile.overview())

### 2.2 검색 단위 구성: 테이블 카드(`table_card`)와 지표 카드(`metric_card`)

문서 RAG에서 긴 문서를 청크(Chunk) 단위로 나누어 색인하듯, Text2SQL에서는 **테이블 1개당 설명 카드 1장(`table_card`), 지표 1개당 공식 카드 1장(`metric_card`)** 으로 구성해 검색 단위로 사용합니다.  
수천~수만 건의 실제 데이터 행을 벡터화하는 것이 아니라, **스키마 구조와 업무 규칙이 정리된 카드(총 13장)만 색인**하는 방식입니다. 앞서 예시로 살펴본 `expense_transactions` 테이블 카드와 `budget_burn_rate`(프로젝트 예산 집행률) 지표 카드가 실제로 어떻게 구성되어 있는지 출력해 보겠습니다.

In [ ]:
print("🃏 [테이블 카드 예시: expense_transactions]")
print(profile.table_card("expense_transactions"))

print("\n" + "=" * 75)
print("📐 [지표 카드 예시: budget_burn_rate]")
print(profile.metric_card("budget_burn_rate"))

출력된 카드를 보면 DB에서 자동 추출한 `[허용값: '승인완료', '결재대기', '반려']`, `조인 경로(FK)`, `샘플 행` 정보 위에 사람이 작성한 `동의어`, `'실집행액은 승인완료만 합산'` 규칙, 그리고 정확한 `CASE WHEN` 집행률 SQL 공식이 한 장의 문서로 통합되어 있음을 확인할 수 있습니다.

### 2.3 운영 안전장치: 물리 DB와 YAML 간 스키마 불일치 자동 감지

실무에서는 DB 컬럼명이 변경되었는데 시멘틱 레이어(`yaml`) 파일 수정을 누락해 오류가 발생하는 경우가 종종 있습니다.  
이를 방지하기 위해 `_apply_table_spec()`은 프로필을 로드할 때 실제 DB에 존재하지 않는 테이블이나 컬럼명이 YAML에 적혀 있으면 즉시 `ValueError`를 발생시켜 실행 전에 불일치를 알려줍니다.

In [ ]:
from rag.sql.semantic_layer import _apply_table_spec

try:
    # 실제 projects 테이블에는 없는 'spent_budget_krw' 컬럼을 시멘틱 레이어에 잘못 적은 상황 시뮬레이션
    broken_spec = {
        "projects": {
            "columns": {
                "spent_budget_krw": "실집행액 (DB에는 존재하지 않는 컬럼)"
            }
        }
    }
    _apply_table_spec(profile.tables, broken_spec)
except ValueError as e:
    print(f"🛡️ 스키마 불일치 즉시 감지 성공! -> ValueError: {e}")

---
## 🧭 Part 3. 스키마 링킹 (Schema Linking): 질문에 필요한 테이블과 조인 경로만 골라내기

### 1) 어떤 문제를 해결하기 위한 것인가?

현업 ERP나 데이터웨어하우스에는 수십~수백 개의 테이블이 존재합니다. 모든 테이블의 상세 카드를 매번 프롬프트에 넣으면 입력 토큰 비용이 커질 뿐 아니라, 이름이 비슷한 컬럼들(`projects.status` 과제 진행상태 vs `expense_transactions.approval_status` 전표 결재상태, 프로젝트 예산 vs 부서 운영 예산)이 한꺼번에 섞여 모델이 잘못된 테이블을 선택하기 쉽습니다. 반대로 테이블을 너무 적게 넣으면 `JOIN`에 꼭 필요한 연결 테이블이 빠져 쿼리를 작성할 수 없습니다.

### 2) 어떻게 작동하는가? (카드 하이브리드 검색 + 상위 방향 FK 확장)

비유하자면 수백 권의 장부가 꽂힌 서고에서 **이번 질문에 관련된 장부 몇 권만 꺼내 오되, 장부 안에 코드 번호(`사번`, `과제코드`)만 적혀 있다면 그 코드의 실제 이름을 알려주는 기준 장부(마스터 테이블)까지 한 세트로 챙겨 오는 과정**입니다.

이를 수행하는 `SchemaRetriever`(`rag/sql/schema_retriever.py`)는 `3_Vector_and_Graph_RAG.ipynb`에서 문서 청크를 찾을 때 사용했던 **`HybridRetriever`와 동일한 검색 구조**에 외래키(FK) 확장 단계를 결합합니다.

| 단계 | 문서 RAG (`3_Vector_and_Graph_RAG` · `HybridRetriever`) | 스키마 링킹 (`4_Text2SQL` · `SchemaRetriever`) |
|:---:|:---|:---|
| **검색 대상** | 정규화된 규정·백서 청크 (`Document`) | **테이블 카드(7장) + 지표 카드(6장) = 총 13장 (`Document`)** |
| **1단계** | `DomainDictionary` 사내 약어 치환 | **동일** (`"클운팀"` → `"클라우드운영팀"`, `"출장비"` → `"국내 여비 및 교통비"`) |
| **2단계** | `BM25` + `Dense` 병렬 검색 | **동일** (단, 키워드가 전혀 겹치지 않는 `BM25 점수 = 0` 카드는 제외) |
| **3단계** | `RRF` 순위 융합 | **동일** + 지표 카드가 선택되면 해당 지표 계산에 필요한 테이블을 자동 포함 |
| **4단계** | `ContextualWindow` (앞뒤 인접 청크 결합) | **상위(부모) 방향 FK 1단계 확장 (`expand`, `fk_hops=1`)** |

### 3) 구체적 사례: 왜 외래키(FK)를 '상위(부모) 방향'으로만 확장할까요?

외래키(FK)에는 참조하는 방향이 있습니다. 예를 들어 지출 전표 테이블(`expense_transactions`)에는 기안자의 이름이나 과제명이 직접 적혀 있지 않고, 사번(`requester_emp_id = 'EMP-2019-001'`)과 과제 코드(`project_id = 'P-01'`)만 저장되어 있습니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    subgraph CHILD["하위 내역 테이블 (검색된 테이블)"]
        TX["expense_transactions (지출 전표)<br/>• requester_emp_id = 'EMP-2019-001'<br/>• project_id = 'P-01'<br/>(사번·코드만 있고 이름 컬럼은 없음)"]
    end

    subgraph PARENT["상위 마스터 테이블 (참조 대상 부모)"]
        EMP["employees (임직원 마스터)<br/>• emp_id = 'EMP-2019-001'<br/>• name = '김철수', rank = '수석'"]
        PRJ["projects (전략 프로젝트)<br/>• project_id = 'P-01'<br/>• project_name = '클라우드 네이티브...'"]
    end

    OTHER["project_members / departments 등<br/>employees를 참조하는 다른 하위 테이블들"]

    TX ==>|"✅ 상위(부모) FK 확장 수행<br/>사번·코드의 실제 이름 조회에 필수"| EMP
    TX ==>|"✅ 상위(부모) FK 확장 수행"| PRJ
    EMP -.->|"❌ 하위(자식) 역방향 확장 차단<br/>직원 명단 조회 시 무관한 테이블 전체 유입 방지"| OTHER

    style TX fill:#e3f2fd,stroke:#1e88e5,stroke-width:2px
    style EMP fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
    style PRJ fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
    style OTHER fill:#ffebee,stroke:#c62828,stroke-width:1px
```

- **상위(부모) 방향 확장 (`✅ 수행`)**: *"결재대기 중인 출장 품의 내역을 보여줘"* 라는 질문으로 `expense_transactions`만 검색되었더라도, 결과 표에 사번 대신 **기안자 이름·직급(`employees`)** 과 **과제명(`projects`)** 을 함께 보여주려면 전표가 참조하는 부모 마스터 테이블들이 반드시 같이 있어야 `JOIN`할 수 있습니다.
- **하위(자식) 방향 확장 (`❌ 차단`)**: 반대로 *"수석 직급 직원 명단을 알려줘"* 라는 질문으로 `employees`가 검색되었을 때, 사번을 참조하는 하위 테이블까지 역방향으로 모두 따라가면 지출 전표(`expense_transactions`), 과제 투입 인력(`project_members`), 프로젝트(`projects`), 부서(`departments`)까지 **질문과 무관한 테이블이 거의 전부 딸려 오기 때문**입니다.

### 3.1 `SchemaRetriever` 검색 및 상위 FK 확장 실험

질문 표현에 따라 **프로젝트 예산(`projects` + `expense_transactions`)** 과 **부서 운영 예산(`department_budgets`)** 이 어떻게 구분되어 선택되는지, 그리고 전표 테이블만 검색되었을 때 부모 테이블(`employees`, `projects`)이 자동으로 추가되는지 확인해 보겠습니다.

In [ ]:
from rag.sql import SchemaRetriever

schema_retriever = SchemaRetriever(profile, embeddings=embeddings, fk_hops=1)
print(f"🗂️ 색인된 스키마 카드 총 {len(schema_retriever.cards)}장 (테이블 {len(profile.tables)} + 지표 {len(profile.metrics)})\n")

test_queries = [
    "진행 중인 프로젝트 중 집행률이 80%를 넘는 과제를 알려줘",
    "부서별 운영비 예산 편성액과 집행 현황을 보여줘",
    "승인 대기 중인 출장 품의 내역을 보여줘",
]

for q in test_queries:
    res = schema_retriever.search(q, top_k=3)
    print(f"🔎 질문: {q}")
    print(f"  1) 약어 정규화 : {res.normalized_query}")
    print(f"  2) 직접 검색됨 : {res.retrieved}")
    print(f"  3) 선택된 지표 : {res.metrics}")
    print(f"  4) 최종 테이블 (FK 1-hop 확장 포함): {res.tables}")
    print("-" * 75)

### 3.2 스키마 링킹만으로는 남는 두 가지 빈틈

위 실행 결과를 자세히 보면, 스키마 검색만으로는 아직 정확한 SQL을 작성하기 어려운 두 가지 상황이 남아 있습니다.

1. **2단계(2-hop) 너머에 있는 조건 테이블 누락**:
   - 지출 전표(`expense_transactions`)에는 기안자 사번만 있고, 부서 코드는 임직원(`employees`) 테이블에 있으며, 정식 부서명(`'클라우드운영팀'`)은 부서(`departments`) 테이블에 있습니다 (`expense_transactions ➔ employees ➔ departments`, **2단계 거리**).
   - 따라서 `fk_hops=1`(1단계 부모 확장)만 적용하면 `['expense_transactions', 'projects', 'employees']`까지만 포함되고 `departments`는 빠집니다. 만약 사용자가 `"클운팀의 결재대기 출장 품의"`처럼 `'부서'`라는 단어 없이 부서 약칭(`'클운팀'`)만 언급하면 `departments` 테이블이 스키마 후보에서 누락될 수 있습니다.
2. **컬럼 안에 실제로 저장된 문자열(Literal) 미확인**:
   - 테이블 카드에 포함된 샘플 행은 2줄뿐입니다. 따라서 질문에 등장한 `'샌프란'`, `'정수석'`, `'클운팀'` 같은 표현이 **어느 테이블의 어느 컬럼에 정확히 어떤 글자로 저장되어 있는지**는 스키마 카드만 보고 알 수 없습니다.

이 두 가지 빈틈을 보완하는 단계가 바로 **Part 4의 값 연결(Value Linking)** 입니다.

In [ ]:
# 확인: expense_transactions 단일 테이블에서 fk_hops=1 확장을 수행하면?
one_hop_tables = schema_retriever.expand(["expense_transactions"])
print("📌 expense_transactions의 1-hop 상위 FK 확장 결과:", one_hop_tables)
print("   👉 기안자의 소속 부서명(departments.dept_name)까지 가려면 2-hop이므로 'departments'가 아직 없습니다!")

---
## 🔗 Part 4. 값 연결 (Value Linking): 일상 표현을 DB 실제 셀 값에 매칭하기

### 1) 어떤 문제를 해결하기 위한 것인가?

문서 벡터 검색에서는 질문에 `'클운팀'`이나 `'샌프란'`이라고 적혀 있어도 의미가 비슷한 `'클라우드운영팀'`, `'샌프란시스코'` 문단을 유연하게 찾아냅니다.  
하지만 SQL의 `WHERE` 조건절(`WHERE dept_name = '클운팀'`)은 **실제 셀에 저장된 문자열과 글자가 정확히 일치하지 않으면 `FALSE`로 판정되어 `0행`이 반환**됩니다(Part 1 실험 1의 실패 원인). 또한 동일한 단어(`'국내출장'`)가 전표 비목(`expense_transactions.category`)과 부서 예산 비목(`department_budgets.category`) 등 여러 테이블에 동시에 존재할 수도 있습니다.

**값 연결(Value Linking)** 은 DB의 주요 컬럼에 실제로 저장된 고유 셀 값들을 미리 색인해 두었다가, 질문 속 단어를 대조하여 **`"질문의 '클운팀'은 departments.dept_name = '클라우드운영팀' 조건으로 작성하세요"`** 처럼 정확한 컬럼과 리터럴 값을 프롬프트에 알려주는 기법입니다.

### 2) 어떻게 작동하는가? (`ValueLinker`의 3가지 동작 원리와 구체 사례)

비유하자면 현업에서 부르는 줄임말이나 일상 표현을 실제 장부 칸에 적힌 **정식 명칭·코드로 찾아 주는 색인 사전**입니다. `rag/sql/value_linker.py`의 `ValueLinker`는 다음 세 가지 원칙으로 동작합니다.

#### ① 조회 조건용 컬럼만 선택 색인 (노이즈 차단)
금액(`48000000`), 날짜(`2026-01-15`), 긴 전표 적요(`description`)까지 모두 색인하면 질문 속 일반 숫자나 조사까지 엉뚱한 컬럼에 매칭됩니다. 따라서 **DDL의 `CHECK` 제약에 정의된 상태·구분 허용값**(`'승인완료'`, `'결재대기'`, `'수석'` 등)과 **`semantic_layer.yaml`에서 `value_lookup: true`로 지정한 명칭·식별자 컬럼**(`부서명, 직원 성명, 과제명, 출장 도시` 등)의 고유값(`SELECT DISTINCT`)만 선별해 색인합니다.

#### ② 사내 약어 치환 후 '최장 일치 우선(Longest-Span Match)' 적용
먼저 `DomainDictionary`로 `'클운팀'` → `'클라우드운영팀'`, `'샌프란'` → `'샌프란시스코'`처럼 사내 약어를 표준 명칭으로 치환한 뒤 DB 셀 값을 찾습니다. 이때 한국어 데이터에서 자주 발생하는 **부분 문자열 중복 매칭 함정**을 피하기 위해 최장 일치 규칙을 적용합니다.

실제로 우리 ERP의 `employees` 테이블에는 **성명이 `'정수석'`인 직원(`name = '정수석'`, 인프라엔지니어링팀장)** 과 **직급을 뜻하는 `'수석'`(`rank = '수석'`)** 이 모두 들어 있습니다. 만약 `"정수석이 총괄 PM인 과제와 수석 직급 인원수를 알려줘"`라는 질문을 단순 포함 여부로만 매칭하면, 앞쪽 이름 `'정수석'` 안에 포함된 `'수석'` 글자까지 중복으로 잡히게 됩니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    Q["질문 원문<br/>'정수석이 총괄 PM인 과제와<br/>수석 직급 인원수를 알려줘'"]

    S1["앞쪽 구간 [0:3] '정수석' (3글자)<br/>➔ employees.name = '정수석' ✅ 채택"]
    S2["앞쪽 내부 [1:3] '수석' (2글자)<br/>➔ '정수석' 구간 안에 완전히 겹침 ❌ 제외"]
    S3["뒤쪽 구간 [16:18] '수석' (2글자)<br/>➔ 겹치지 않는 독립 구간 ✅ employees.rank = '수석' 채택"]

    Q --> S1
    Q --> S2
    Q --> S3

    style S1 fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
    style S2 fill:#ffebee,stroke:#c62828,stroke-width:1.5px
    style S3 fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

이처럼 **더 긴 문자열(`'정수석'`)이 매칭된 구간 안에 완전히 포함되는 짧은 문자열(`'수석'`)은 버리고**, 별도의 위치에서 독립적으로 등장한 `'수석'`만 직급 조건으로 살려 정확한 힌트를 생성합니다.

#### ③ 다중 컬럼 후보 안내 및 누락 테이블 자동 역보강 (Schema Backfill)
- **다중 컬럼 안내**: `'국내출장'`처럼 `expense_transactions.category`와 `department_budgets.category` 두 테이블에 모두 존재하는 값은 두 컬럼 후보를 함께 제시하여, SQL 생성 모델이 질문 맥락(전표 조회인지 부서 예산 조회인지)에 맞는 쪽을 선택하도록 돕습니다.
- **누락 테이블 역보강**: 앞서 Part 3.2에서 `"클운팀의 결재대기 국내출장 품의"` 질문은 전표 테이블(`expense_transactions`)에서 1단계 부모만 펼치면 `departments` 테이블이 빠진다고 확인했습니다. 이때 `ValueLinker`가 `'클운팀' ➔ departments.dept_name = '클라우드운영팀'`을 찾아내면, 파이프라인이 *"조건절에 쓰려면 `departments` 테이블도 필요하구나"* 라고 판단하여 **빠져 있던 `departments` 테이블 카드를 스키마 컨텍스트에 자동으로 보강**합니다.

### 4.1 `ValueLinker` 매칭 실험 및 스키마 역보강 확인

In [ ]:
from rag.sql import ValueLinker

value_linker = ValueLinker(profile)
print(f"📇 ValueLinker 색인 완료: 총 {len(value_linker.index)}개의 고유 셀 값 색인됨")

sample_keys = ["클라우드운영팀", "정수석", "수석", "국내출장", "샌프란시스코"]
for k in sample_keys:
    print(f"  • '{k}' -> {value_linker.index.get(k)}")

print("\n" + "=" * 75)
vl_questions = [
    "클운팀의 결재대기 국내출장 품의 금액 합계는?",
    "정수석이 총괄 PM인 과제와 수석 직급 인원수를 알려줘",
    "샌프란 출장 전표 내역 보여줘",
]

for q in vl_questions:
    matches = value_linker.link(q)
    print(f"❓ 질문: {q}")
    print(ValueLinker.render(matches))
    print("-" * 75)

In [ ]:
# 스키마 링킹(Part 3) + 값 연결(Part 4) 상호 보완 시연:
# expense_transactions만 선택되어 'departments'가 빠진 상태에서 "클운팀 결재대기 국내출장" 값 연결이 일어나면?
selected_tables = schema_retriever.expand(["expense_transactions"])  # ['expense_transactions', 'projects', 'employees']
matches = value_linker.link("클운팀의 결재대기 국내출장 품의 금액 합계는?")

value_tables = [col.split(".")[0] for m in matches for col in m.columns]
missing_tables = [t for t in dict.fromkeys(value_tables) if t not in selected_tables]

print(f"1️⃣ 스키마 검색 직후 테이블 : {selected_tables}")
print(f"2️⃣ 값 연결이 찾아낸 테이블 : {list(dict.fromkeys(value_tables))}")
print(f"3️⃣ 누락 감지되어 보강될 테이블 : {missing_tables}")

if missing_tables:
    selected_tables += [t for t in schema_retriever.expand(missing_tables) if t not in selected_tables]
print(f"4️⃣ 보강 완료된 최종 테이블 : {selected_tables}")

---
## 🛡️ Part 5. 읽기 전용 2중 가드레일과 자가 수정(Self-Healing) 워크플로우

시멘틱 레이어·스키마 링킹·값 연결로 정확한 프롬프트 재료를 준비했더라도, **LLM이 생성한 SQL을 검증 없이 DB에 바로 실행해서는 안 됩니다.**  
실수로 데이터를 변경·삭제하는 쿼리가 생성되거나, 컬럼명 오타·조인 실수로 런타임 오류가 발생할 수 있기 때문입니다. 이를 막기 위해 `rag/sql/executor.py`와 `rag/sql/pipeline.py`는 **2중 읽기 전용 안전장치**와 **자가 수정(Self-Healing) 루프**를 적용합니다.

### 5.1 읽기 전용 2중 가드레일 (`rag/sql/executor.py`)

| 방어 단계 | 구현 방식 | 방어하는 위험 및 설계 포인트 |
|:---|:---|:---|
| **1차: SQL 문자열 사전 검수**<br/>(`validate_sql`) | • 작은따옴표 문자열(`'...삭제...'`)과 주석을 먼저 분리해 **정상 조회 쿼리 오탐 방지**<br/>• 단일 `SELECT` / `WITH` 시작 강제, 세미콜론(`;`) 다중 문장 실행 차단<br/>• `INSERT`, `UPDATE`, `DELETE`, `DROP`, `PRAGMA`, `ATTACH` 등 변경·관리 명령어 차단 | 데이터 위변조·삭제 시도 및 다중 쿼리 주입(SQL Injection) 사전 차단 |
| **2차: DB 엔진 권한·자원 제한**<br/>(`connect_readonly`, `run_readonly_query`) | • SQLite 연결 시 `?mode=ro`(읽기 전용 파일 모드) 및 `PRAGMA query_only = ON` 강제<br/>• `set_progress_handler`로 **쿼리 실행 시간 제한(5초)** 적용<br/>• 결과는 최대 **50행(`max_rows`)** 까지만 잘라 반환하되 전체 건수는 함께 표시 | 정규식 검수를 우회한 쓰기 시도 원천 차단, 잘못된 조인으로 인한 DB 지연 및 프롬프트 토큰 초과 방지 |

> 💡 **문자열 오탐 방지 예시**: 전표 적요(`description`)에 `'레거시 백업본 삭제 용역'` 같은 문구가 들어 있을 때 사용자가 `WHERE description LIKE '%삭제%'` 또는 영문 로그 `'%DELETE%'`를 검색하는 것은 정상적인 읽기 작업입니다. `validate_sql()`은 따옴표 안의 문자열 리터럴을 먼저 지운 뒤 나머지 SQL 구문에서만 금지 키워드를 검사하므로 정상 조회가 차단되지 않습니다.

또한 쿼리 실행 중 컬럼명 오타 등으로 SQLite 오류가 발생하더라도 파이썬 예외로 중단되지 않고, `ExecutionResult(ok=False, error=...)` 객체에 **DB의 오류 원인을 담아 반환**합니다. 덕분에 다음 단계에서 LLM이 이 오류 메시지를 보고 쿼리를 스스로 수정할 수 있습니다.

In [ ]:
from rag.sql import validate_sql, run_readonly_query

guardrail_cases = [
    ("쓰기 명령 차단 (DELETE)", "DELETE FROM expense_transactions WHERE tx_id = 'TX-2026-0001'"),
    ("다중 문장 차단 (; DROP)", "SELECT project_id FROM projects; DROP TABLE projects"),
    ("문자열 내부 키워드 허용 (오탐 방지)", "SELECT tx_id, description FROM expense_transactions WHERE description LIKE '%삭제%'"),
    ("없는 컬럼 조회 (Self-Healing용 에러 반환)", "SELECT project_id, spent_budget_krw FROM projects"),
]

for title, sql in guardrail_cases:
    cleaned, v_err = validate_sql(sql)
    exec_res = run_readonly_query(ERP_DB_PATH, sql)
    print(f"🧪 [{title}]")
    print(f"  • SQL        : {sql}")
    print(f"  • 1차 검증   : {'통과 (None)' if v_err is None else f'차단 -> {v_err}'}")
    print(f"  • 실행 결과  : ok={exec_res.ok} | {exec_res.summary() if exec_res.ok else exec_res.error}")
    print("-" * 75)

### 5.2 `Text2SQL` LangGraph 워크플로우 조립 (`rag/sql/pipeline.py`)

이제 앞서 살펴본 모든 모듈을 하나의 LangGraph 상태 그래프(`Text2SQL`)로 연결합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    ST(["질문 입력"]) --> RS["1. retrieve_schema<br/>테이블·지표 카드 검색<br/>+ 상위 부모 FK 확장"]
    RS --> LV["2. link_values<br/>DB 실제 셀 값 매칭<br/>+ 누락 테이블 역보강"]
    LV --> GS["3. generate_sql<br/>구조화 출력 (SQLDraft)<br/>sql · assumptions · needs_clarification"]

    GS -->|"모호한 질문 감지<br/>(needs_clarification = True)"| ASK(["되묻기 반환<br/>해석 선택지 안내"])
    GS -->|"단일 SQL 생성"| EX["4. execute_sql<br/>2중 가드레일 읽기 실행"]

    EX -->|"SQL 오류 발생 또는 첫 0행 반환<br/>(오류·재점검 피드백 주입, 최대 3회)"| GS
    EX -->|"실행 및 검증 완료"| ED(["최종 결과 반환<br/>(표 + 적용 가정 명시)"])

    style RS fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style LV fill:#fff3e0,stroke:#fb8c00,stroke-width:1.5px
    style GS fill:#ede7f6,stroke:#5e35b1,stroke-width:1.5px
    style EX fill:#ffebee,stroke:#d32f2f,stroke-width:1.5px
    style ASK fill:#fff8e1,stroke:#f57f17,stroke-width:1.5px
    style ED fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

이 워크플로우에는 Part 1에서 보았던 실패를 막아주는 **두 가지 핵심 분기**가 들어 있습니다.

1. **모호성 감지 및 되묻기 분기 (`_after_generate`)**:
   - 질문이 시멘틱 레이어의 기본 규칙만으로는 하나로 결정되지 않을 때(예: Part 1 실험 2였던 *"예산 집행률이 80% 이상인 항목"* — 부서 운영 예산인지 전략 프로젝트 예산인지 불분명), 임의로 한쪽 테이블만 골라 조회하는 대신 `needs_clarification=True`와 가능한 해석 선택지(`candidates`)를 담아 사용자나 상위 에이전트에게 되묻습니다.
   - 반대로 기본 규칙(예: *"실집행액은 승인완료 건만 합산"*, *"'클라우드팀'은 '클라우드운영팀'으로 해석"*)을 적용해 쿼리를 작성한 경우에는 `assumptions`(적용한 가정) 필드에 그 기준을 함께 적어 결과의 근거를 투명하게 밝힙니다.
2. **오류 복구 및 `0행` 재검증 루프 (`_after_execute`, Self-Healing)**:
   - **SQL 오류 발생 시**: SQLite가 반환한 에러 메시지(예: `no such column: spent_budget_krw`)를 피드백으로 넣어 올바른 컬럼과 `JOIN` 구조로 다시 작성하게 합니다.
   - **첫 실행 결과가 `0행`일 때 1회 재점검 (`empty_checked`)**: 결과가 `0행`이면 실제로 데이터가 없는 것일 수도 있지만, `WHERE` 절의 문자열이나 조건을 잘못 적었을 가능성도 있습니다. 따라서 첫 `0행`이 나오면 곧바로 종료하지 않고 *"조건 값이 DB 실제 저장 값과 일치하는지 한 번 더 확인하고, 실제로 조건에 맞는 행이 없는 것이 맞다면 동일 SQL을 유지하라"* 고 1회 재검증을 거칩니다.

### 5.3 Part 1 실패 질문 재실험: 원시 DDL 방식 vs `Text2SQL` 파이프라인 비교

Part 1에서 `0행`과 반쪽짜리 결과를 반환했던 동일한 두 질문, 그리고 대상을 `'전사 전략 프로젝트'`로 명확히 지정한 후속 질문(시나리오 A)을 `Text2SQL`에 차례로 실행해 결과를 비교해 보겠습니다.

In [ ]:
from rag.sql import Text2SQL

erp_t2s = Text2SQL.from_semantic_layer(
    SEMANTIC_LAYER_PATH,
    llm=llm,
    db_path=ERP_DB_PATH,
    embeddings=embeddings,
)

# 1. Part 1에서 0행이 나왔던 질문 재도전 ('클라우드팀' -> '클라우드운영팀' 매핑 및 가정 명시)
res_cloud = erp_t2s.invoke("클라우드팀의 연간 운영 예산과 비목별 편성액을 알려줘.")
print("🎯 [재도전 1: '클라우드팀' 예산 조회]")
print(res_cloud.to_text())

print("\n" + "=" * 75)

# 2. Part 1에서 한쪽 테이블만 멋대로 조회했던 모호한 질문 재도전 -> 되묻기(needs_clarification) 작동!
res_ambig = erp_t2s.invoke("예산 집행률이 80% 이상인 항목을 모두 알려줘.")
print("🎯 [재도전 2: 모호한 '예산 집행률 80% 이상' 질문 -> 조용한 오답 대신 되묻기!]")
print(res_ambig.to_text())

print("\n" + "=" * 75)

# 3. 사용자가 후보 중 '전사 전략 프로젝트'를 선택해 구체화한 경우 (시나리오 A)
res_scen_a = erp_t2s.invoke(
    "현재 진행 중인 전사 전략 프로젝트 중 예산 집행률이 80%를 초과하는 프로젝트의 "
    "코드, 명칭, 주관 부서, 총괄 PM, 승인 예산, 실집행액, 집행률(%)을 구해줘."
)
print("🎯 [시나리오 A: 전사 전략 프로젝트 집행률 80% 초과 조기경보 검출]")
print(res_scen_a.to_text())

### 5.4 그래프 내부 동작 추적: `0행` 재검증 및 오류 자가 수정 과정 확인

`Text2SQL` 그래프 내부에서 피드백 루프가 실제로 어떻게 동작하는지 단계별 로그로 확인해 보겠습니다.
1. **실제 `0행`인 질문의 2회 검증 흐름**: 우리 DB에서 프로젝트 최고 집행률은 `86.0%(P-02)`이므로, *"집행률이 95%를 초과하는 전략 프로젝트"* 는 실제로 존재하지 않습니다. 이 질문을 던지면 `generate_sql(1차)` ➔ `execute_sql(0행 감지)` ➔ `generate_sql(2차 조건 재점검)` ➔ `execute_sql(0행 최종 확정)` 순서로 신중하게 확인한 뒤 종료됩니다.
2. **존재하지 않는 컬럼 조회 시 자가 수정**: 1차 시도에서 모델이 `projects` 테이블에 없는 `spent_budget_krw` 컬럼을 잘못 조회해 `no such column` 에러가 발생했다고 가정하고, 피드백을 받은 `_generate_sql` 노드가 2차 시도에서 `expense_transactions` 조인 쿼리로 스스로 고쳐 내는 과정을 확인합니다.

In [ ]:
# 1) 0행 발생 시 자동 재검증(empty_checked) 그래프 스트리밍 추적
zero_row_q = "현재 진행 중인 전사 전략 프로젝트 중 예산 집행률이 95%를 초과하는 프로젝트를 알려줘."
print(f"🔄 [스트리밍 추적] 질문: {zero_row_q}\n")

for step_num, event in enumerate(erp_t2s.graph.stream({"question": zero_row_q, "context": "", "attempts": 0}), 1):
    node_name, node_out = next(iter(event.items()))
    if node_name == "retrieve_schema":
        print(f"  Step {step_num} [{node_name}] 선택 테이블: {node_out['selected_tables']} | 지표: {node_out['selected_metrics']}")
    elif node_name == "link_values":
        print(f"  Step {step_num} [{node_name}] 값 힌트:\n    " + node_out["value_hints"].replace("\n", "\n    "))
    elif node_name == "generate_sql":
        print(f"  Step {step_num} [{node_name}] (시도 {node_out['attempts']}회차) SQL: {node_out['draft'].sql}")
    elif node_name == "execute_sql":
        r = node_out["result"]
        fb_msg = "재검증 피드백 발생! -> generate_sql로 루프" if node_out.get("feedback") else "검증 완료 -> 종료"
        print(f"  Step {step_num} [{node_name}] 결과: {r.summary()} | {fb_msg}")

print("\n" + "=" * 75)

# 2) SQL 오류 피드백을 통한 쿼리 자가 수정(Self-Correction) 시뮬레이션
q_fix = "프로젝트별 승인 예산과 실집행액을 조회해줘."
schema_res = erp_t2s.schema_retriever.search(q_fix, top_k=3)
error_state = {
    "question": q_fix,
    "context": "",
    "schema_context": schema_res.context,
    "value_hints": ValueLinker.render(erp_t2s.value_linker.link(q_fix)),
    "attempts": 1,
    "feedback": (
        "이전 SQL:\nSELECT project_id, project_name, approved_budget_krw, spent_budget_krw FROM projects\n"
        "문제: SQLite 오류: no such column: spent_budget_krw\n오류를 고친 SQL을 다시 작성하세요."
    ),
}
healed_out = erp_t2s._generate_sql(error_state)
healed_exec = run_readonly_query(ERP_DB_PATH, healed_out["draft"].sql)
print("🩹 [SQL 오류 피드백 기반 자가 수정 결과]")
print(f"  • 주입된 오류 피드백 : SQLite 오류: no such column: spent_budget_krw")
print(f"  • 수정된 SQL (2회차) : {healed_out['draft'].sql}")
print(f"  • 재실행 결과 ({healed_exec.summary()}) :\n{healed_exec.to_markdown()}")

---
## 🤝 Part 6. 에이전트 도구로 결합하기: 비정형 사내 규정 × 정형 ERP 복합 감사

실무 질문 중에는 **비정형 문서(사내 규정집)** 와 **정형 데이터베이스(ERP 전표)** 양쪽을 모두 확인해야 답할 수 있는 경우가 많습니다.  
예를 들어 *"여비 규정의 숙박비 한도를 위반한 결재대기 출장 전표를 찾아줘"* 라는 질문을 생각해 보겠습니다.
- **숙박비 한도 기준(직급별 · 출장 지역별 1박 상한액)**: 사내 규정 마크다운 문서(`국내여비교통비지급규정 제14조`)에만 적혀 있고, ERP DB에는 저장되어 있지 않습니다.
- **실제 출장 청구 내역(기안자 직급, 출장지, 박수, 청구 금액, 결재 상태)**: 규정 문서에는 없고, ERP DB(`expense_transactions`, `employees`)에만 저장되어 있습니다.

이 두 세계를 깔끔하게 연결하기 위해 `app/agents/tool_rag_agent/tools.py`의 `search_enterprise_erp` 도구는 `query`와 `context` 두 개의 인자를 받도록 설계되어 있습니다.

```python
@tool
def search_enterprise_erp(query: str, context: str = "") -> str:
    """사내 ERP 정형 데이터베이스에서 실시간 수치와 현황 레코드를 조회합니다.
    query에는 SQL 문이 아니라 자연어 질문을 넣으세요. SQL은 이 도구가 스키마를 보고 직접 생성합니다."""
    return _erp_text2sql().invoke(query, context=context).to_text()
```

```mermaid
sequenceDiagram
    participant U as 사용자
    participant AG as 메인 ReAct 에이전트<br/>(DB 스키마·SQL 모름)
    participant POL as search_company_policy<br/>(비정형 규정 검색)
    participant ERP as search_enterprise_erp<br/>(Text2SQL 파이프라인)

    U->>AG: 여비규정 제14조 한도 위반 결재대기 국내출장 전표 조회 요청
    AG->>POL: ① "국내 여비 규정 제14조 직급별 지역별 숙박비 상한액"
    POL-->>AG: 직급·지역별 1박 상한표 반환 (예: 수석 서울 15만, 책임 부산 13만 등)
    AG->>ERP: ② query="결재대기 국내출장 중 1박 숙박비 한도 초과 전표 조회"<br/>context="제14조 직급·지역별 1박 상한액 기준값..."
    Note over ERP: 스키마 링킹 + 값 연결 후<br/>context 기준값을 SQL CASE WHEN으로 변환해 실행
    ERP-->>AG: 한도 초과 전표 목록 (전표번호, 기안자, 청구액, 초과액) 반환
    AG-->>U: ③ 규정 조항 근거와 위반 전표 감사 표 종합 보고
```

이렇게 설계했을 때 얻는 실무적 이점은 두 가지입니다.
1. **메인 에이전트의 프롬프트 단순화 (관심사 분리)**: 메인 에이전트는 ERP에 어떤 테이블과 컬럼이 있는지, SQL을 어떻게 짜야 하는지 전혀 알 필요가 없습니다. 자연어 질문(`query`)만 넘기면 스키마 검색·값 연결·SQL 검증은 `search_enterprise_erp` 내부의 `Text2SQL` 파이프라인이 전담합니다.
2. **`context` 인자를 통한 외부 규정 수치의 동적 반영**: 메인 에이전트가 먼저 `search_company_policy`로 읽은 **제14조 직급·지역별 숙박비 상한액**을 `context` 문자열로 전달하면, `Text2SQL`이 그 기준값을 읽어 SQL의 `CASE WHEN` 조건식에 반영합니다. 나중에 여비 규정의 한도 금액이 개정되더라도 Text2SQL 코드나 DB 시멘틱 레이어를 수정할 필요가 없습니다.

### 6.1 실습: 여비 규정(제14조) 위반 '결재대기' 국내출장 전표 적발 (시나리오 B)

`search_company_policy`와 `search_enterprise_erp` 두 도구를 연결한 감사 에이전트를 생성하고, 규정 문서 조회부터 ERP 위반 전표 검출까지 이어지는 과정을 실행해 보겠습니다.

In [ ]:
from langchain.agents import create_agent
from app.agents.tool_rag_agent.tools import search_company_policy, search_enterprise_erp

AUDIT_SYSTEM_PROMPT = """당신은 (주)넥스트AI의 내부감사 AI 에이전트입니다.
1. 규정 기준(한도 금액, 전결 기준 등)이 필요한 질문은 반드시 먼저 `search_company_policy`로 정확한 조항과 수치를 확인하세요.
2. 확인한 규정 기준값을 `search_enterprise_erp`의 `context` 인자에 구체적으로 전달하여 ERP 데이터를 조회하세요.
3. 조회된 전표 번호, 기안자 성명·직급·부서, 출장지·박수, 청구 금액, 규정 상한액, 초과 금액을 표로 명확히 정리해 보고하세요."""

audit_agent = create_agent(
    model=llm,
    tools=[search_company_policy, search_enterprise_erp],
    system_prompt=AUDIT_SYSTEM_PROMPT,
)

scenario_b_question = (
    "사내 국내 여비 및 교통비 지급 규정(제14조)의 직급별·지역별 숙박비 상한액을 기준으로, "
    "현재 ERP에서 '결재대기' 중인 '국내출장' 품의 가운데 1박당 숙박비 한도를 초과하여 위반한 내역을 모두 찾아 보고해줘."
)

response = audit_agent.invoke({"messages": [{"role": "user", "content": scenario_b_question}]})

print("🔍 [에이전트 도구 호출 궤적 확인]")
for msg in response["messages"]:
    if getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            print(f"  🛠️ 도구 호출: {tc['name']}")
            for k, v in tc["args"].items():
                preview = v if len(str(v)) <= 120 else str(v)[:120] + "..."
                print(f"     - {k}: {preview}")
    elif msg.type == "tool":
        first_line = str(msg.content).splitlines()[0] if msg.content else ""
        print(f"  📥 도구 응답 ({msg.name}): {first_line} (총 {len(str(msg.content)):,}자)")

print("\n" + "=" * 75)
print("📋 [최종 하이브리드 감사 리포트]\n")
print(normalize_content(response["messages"][-1].content))

---
## 🏁 Part 7. 정리 및 다음 단계

이번 노트북에서는 정형 데이터베이스를 에이전트의 검색 도구로 안전하고 정확하게 연결하기 위한 **Text2SQL 파이프라인 구조**를 단계별로 살펴보았습니다.

### 핵심 구성 요소 요약

| 구성 요소 | 해결한 현업 문제 | 동작 방식 및 구현 모듈 |
|:---|:---|:---|
| **1. 시멘틱 레이어**<br/>*(Semantic Layer)* | 물리 컬럼명만으로는 회사의 집계 기준(예: `'실집행액'` = `'승인완료'`만 합산)과 계산 공식을 알 수 없음 | `DatabaseProfile`: DB 물리 정보 자동 추출(`PRAGMA`) + `semantic_layer.yaml`(용어 설명·규칙·지표 공식) 결합 |
| **2. 스키마 링킹**<br/>*(Schema Linking)* | 전체 테이블을 다 넣으면 혼선이 생기고, 일부만 넣으면 코드·사번을 해석할 마스터 테이블이 빠짐 | `SchemaRetriever`: 약어 치환 ➔ `BM25 + Dense` ➔ `RRF` ➔ **참조 부모 방향 1-hop FK 자동 확장** |
| **3. 값 연결**<br/>*(Value Linking)* | 일상 표현·약어를 그대로 `WHERE` 절에 쓰면 글자 불일치로 `0행`이 반환됨 | `ValueLinker`: 상태·명칭 컬럼 선택 색인 + **최장 일치 매칭**(`'정수석'` vs `'수석'`) + **누락 테이블 역보강** |
| **4. 안전 실행 & 자가 수정**<br/>*(Guardrail & Self-Healing)* | 쓰기 쿼리 위험, 런타임 SQL 오류, 모호한 질문에 대한 임의 추측 오답 | `executor.py` + `Text2SQL`: 정규식·`mode=ro` 2중 차단, 오류·첫 `0행` 재검증 루프, 모호한 질문 **되묻기(`needs_clarification`)** |
| **5. 도구 경계 캡슐화**<br/>*(Tool Encapsulation)* | 사내 규정 문서의 기준값과 DB 전표 데이터를 함께 대조해야 함 | `search_enterprise_erp(query, context)`: SQL 생성은 도구 내부에 숨기고 외부 규정 수치는 `context`로 전달 |

---

### 💪 스스로 해보기 (심화 연습)

위 `audit_agent`에 아래 질문(**시나리오 E: 부서 운영 예산 통제 등급 감사**)을 실행해 보세요.  
이번에는 전략 프로젝트 예산(`projects`)이 아니라 **부서 운영 예산(`department_budgets`)** 테이블과 **부서장·본부장 확인을 위한 임직원 자기참조 조인(`employees.manager_emp_id`)** 이 선택되는 과정을 확인할 수 있습니다.

```python
# 연습용 질문 예시 (새 코드 셀을 만들어 실행해 보세요)
practice_q = (
    "사내 전사위임전결및예산집행규정(제11조)의 집행률 단계별 통제 기준을 먼저 확인한 뒤, "
    "부서 운영 예산 중 집행률이 80% 이상인 비목을 모두 찾아 통제 등급(조기 경보/동결 경계/차단)과 "
    "경보를 받아야 할 부서장 및 소속 본부장을 보고해줘."
)
```

---

### 👉 다음 단계: `5_MCP_and_Skills.ipynb`

지금까지 사내 규정 Vector DB, AWS 기술 백서 DB, 지식 그래프(`3_Vector_and_Graph_RAG.ipynb`), 그리고 사내 ERP Text2SQL(`4_Text2SQL.ipynb`)까지 네 가지 엔터프라이즈 검색 도구를 구축했습니다.  
다음 **`5_MCP_and_Skills.ipynb`** 에서는 이 검색 도구들을 에이전트 코드에서 분리해 독립된 **MCP(Model Context Protocol) 서버**로 구성하고, **인증·권한 제어** 및 **Skills 기반 점진적 도구 탐색** 구조로 확장하는 방법을 다룹니다.